# PUMA - Stage 2: Tissue, context, refinement, and packaging

Run this notebook after Stage 1 is complete. It builds the Stage-2 candidates and cache, trains the tissue model and Stage-2 model, creates OOF predictions, fits risk calibration, and packages the final model files.

**Status: this project is still under development.**


In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Paths and package setup

Find the same project folder used in Stage 1 and install the required package dependencies.


In [2]:
from pathlib import Path
import importlib.util
import subprocess
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/Research/PUMA')
CODE_ROOT_OVERRIDE = None  # Example: PROJECT_ROOT / 'Code' / 'PUMA'


def is_code_root(path: Path | None) -> bool:
    return bool(
        path is not None
        and (path / 'pyproject.toml').is_file()
        and (path / 'src' / 'puma_pipeline' / '__init__.py').is_file()
    )


candidates = [
    CODE_ROOT_OVERRIDE,
    PROJECT_ROOT / 'Version 16',
    PROJECT_ROOT,
    Path.cwd(),
]
CODE_ROOT = next((p.resolve() for p in candidates if is_code_root(p)), None)
if CODE_ROOT is None:
    search_roots = [PROJECT_ROOT / 'Code', PROJECT_ROOT]
    found = []
    for parent in search_roots:
        if not parent.is_dir():
            continue
        for child in parent.iterdir():
            if child.is_dir() and is_code_root(child):
                found.append(child.resolve())
    found = sorted(set(found))
    if len(found) == 1:
        CODE_ROOT = found[0]
if CODE_ROOT is None:
    raise FileNotFoundError(
        'Cannot find the PUMA source tree. Put the extracted project under '
        f'{PROJECT_ROOT / "Code" / "PUMA"} or set CODE_ROOT_OVERRIDE.'
    )

CONFIG_TEMPLATE = CODE_ROOT / 'configs' / 'train_config.json'
if not CONFIG_TEMPLATE.is_file():
    raise FileNotFoundError(f'Missing training config: {CONFIG_TEMPLATE}')

requirements = {
    'timm': 'timm>=1.0.15,<2',
    'huggingface_hub': 'huggingface-hub>=0.23,<1',
    'tifffile': 'tifffile>=2024.2',
    'skimage': 'scikit-image>=0.22,<1',
    'scipy': 'scipy>=1.11,<2',
    'tqdm': 'tqdm>=4.66,<5',
}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing dependencies:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

SRC_ROOT = CODE_ROOT / 'src'
for name in list(sys.modules):
    if name == 'puma_pipeline' or name.startswith('puma_pipeline.'):
        del sys.modules[name]
if str(SRC_ROOT) in sys.path:
    sys.path.remove(str(SRC_ROOT))
sys.path.insert(0, str(SRC_ROOT))

import puma_pipeline

imported_path = Path(puma_pipeline.__file__).resolve()
if not str(imported_path).startswith(str(SRC_ROOT.resolve())):
    raise RuntimeError(f'puma_pipeline was imported from the wrong location: {imported_path}')

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CODE_ROOT    =', CODE_ROOT)
print('package      =', imported_path)


PROJECT_ROOT = /content/drive/MyDrive/Research/PUMA
CODE_ROOT    = /content/drive/MyDrive/Research/PUMA/Version 16
package      = /content/drive/MyDrive/Research/PUMA/Version 16/src/puma_pipeline/__init__.py


## 2. Load the Stage-1 config

Load the resolved config created by the first notebook so both stages use the same paths and feature contract.


In [3]:
import json
from puma_pipeline.config import PumaConfig

RESOLVED_CONFIG = CODE_ROOT / 'configs' / 'resolved_config.json'
if not RESOLVED_CONFIG.is_file():
    base = CODE_ROOT / 'configs' / 'train_config.json'
    payload = json.loads(base.read_text(encoding='utf-8'))
    payload['project_root'] = str(PROJECT_ROOT)
    config = PumaConfig(**payload)
    config.save(RESOLVED_CONFIG)
else:
    config = PumaConfig.load(RESOLVED_CONFIG)
    if (
        config.stage2_feature_contract != 'self_exclusion_pixel_sampling'
        or config.stage2_output_dir != 'PUMA_stage2_outputs'
        or config.cache_dir != 'PUMA_stage2_cache'
    ):
        base = CODE_ROOT / 'configs' / 'train_config.json'
        payload = json.loads(base.read_text(encoding='utf-8'))
        payload['project_root'] = str(PROJECT_ROOT)
        config = PumaConfig(**payload)
        config.save(RESOLVED_CONFIG)
        print('Refreshed the resolved config to match the current Stage-2 contract.')

print('Config:', RESOLVED_CONFIG)
print('Full fingerprint:', config.fingerprint)
print('Stage-1 fingerprint:', config.stage1_fingerprint)


Config: /content/drive/MyDrive/Research/PUMA/Version 16/configs/resolved_config.json
Full fingerprint: 2c63e1ee72de5159
Stage-1 fingerprint: f0d55e624d25ee7d


## 3. Check Stage-1 outputs

Verify the Stage-1 checkpoints and OOF files before building Stage 2.


In [4]:
import torch
from puma_pipeline.store import PumaArtifactStore
from puma_pipeline.stage1 import stage1_oof_paths, stage1_fold_checkpoint, stage1_final_checkpoint
from puma_pipeline.stage2.encoder import ensure_uni2_checkpoint

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU-only runtime detected. Artifact/config checks below are still valid.')
    print('Switch to GPU before running tissue/model inference, UNI2 cache extraction, or training cells.')

uni2_path = ensure_uni2_checkpoint(config.path('uni2_checkpoint'))
print('UNI2-h:', uni2_path)
print('UNI2-h mode: local checkpoint (download fallback is used only when this file is absent)')

store = PumaArtifactStore.open(config.path('artifact_dir'))
print('ROIs:', len(store.images), 'Nuclei:', int(store.offsets[-1]))
for fold in range(config.number_of_folds):
    print(f'fold {fold}:', stage1_fold_checkpoint(config, fold))
print('final Stage 1:', stage1_final_checkpoint(config))
for path in stage1_oof_paths(config):
    assert path.is_file(), path
    print('OOF:', path)


CUDA available: True
GPU: Tesla T4
UNI2-h: /content/drive/MyDrive/Research/PUMA/PUMA_pretrained_checkpoints/uni2_h_model.bin
UNI2-h mode: local checkpoint (download fallback is used only when this file is absent)
ROIs: 205 Nuclei: 97378
fold 0: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_0/stage1_final_ema.pt
fold 1: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_1/stage1_final_ema.pt
fold 2: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_2/stage1_final_ema.pt
fold 3: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_3/stage1_final_ema.pt
fold 4: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_4/stage1_final_ema.pt
final Stage 1: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/final_all_data/stage1_final_ema.pt
OOF: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/stage1_oof_candidates.npy
OOF: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/stage1_oof_gt_features.npy


## 4. Build Stage-2 candidates

Create the candidate table from Stage-1 OOF detections and clean GT candidates. GT polygons are kept for BioMask supervision.


In [5]:
from pprint import pprint
from puma_pipeline.data.audit import audit_oof

pprint(audit_oof(config))


{'clean_gt': 97378,
 'contract': 'Every row inherits the held fold of its ROI; clean GT rows use '
             'only OOF Stage-1 sampled features.',
 'fold_counts': [46630, 47775, 48496, 47221, 46191],
 'real_oof': 138935,
 'rows': 236313}


## 5. Train the tissue model

The tissue decoder uses frozen Stage-1 FPN features and predicts the six tissue classes used as context by Stage 2.


### GPU fallback note

The tissue feature extractor can reduce its micro-batch or use a convolution fallback when the current GPU cannot run the frozen Stage-1 FPN normally.


In [ ]:
import torch
from puma_pipeline.tissue.trainer import build_tissue_targets, generate_tissue_oof

FORCE_TISSUE = False

if not torch.cuda.is_available():
    raise RuntimeError('Stage-2 tissue generation requires a CUDA GPU runtime.')

gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
print(f'GPU: {gpu_name} | compute capability: {gpu_capability[0]}.{gpu_capability[1]}')
if gpu_capability[0] < 8:
    print(
        'Pre-Ampere GPU detected. The tissue trainer will use its persistent '
        'Stage-1 FPN cuDNN fallback automatically if this GPU cannot select a '
        'valid cuDNN engine. This avoids the repeated retry loop seen on Tesla T4.'
    )

print('Tissue targets:', build_tissue_targets(config, force=FORCE_TISSUE))
print('OOF tissue probabilities:', generate_tissue_oof(config, force=FORCE_TISSUE))


## 6. Build the Stage-2 cache

Encode the frozen UNI2-h multi-view appearance features and save the other candidate features used during Stage-2 training.


In [ ]:
from puma_pipeline.stage2.cache import build_stage2_cache

FORCE_STAGE2_CACHE = False
cache_manifest = build_stage2_cache(config, force=FORCE_STAGE2_CACHE)
print('Cache manifest:', cache_manifest)


In [ ]:
from google.colab import runtime
runtime.unassign()

## 7. Train Stage-2 OOF folds

Train the local and graph phases for each fold, then create OOF Stage-2 predictions. The current Stage-2 schedule does not use early stopping.


In [ ]:
import numpy as np

candidate_path = config.path("cache_dir") / "candidates.npy"
c = np.load(candidate_path, mmap_mode="r", allow_pickle=False)

# kind == 0 means detector candidates used by graph training
real = c[c["kind"] == 0]

counts = np.bincount(
    np.asarray(real["roi_index"], dtype=np.int64),
    minlength=config.expected_public_rois,
)

print("Mean candidates / ROI :", counts.mean())
print("Median               :", np.median(counts))
print("95th percentile      :", np.percentile(counts, 95))
print("Max candidates / ROI :", counts.max())
print("Max ROI index        :", counts.argmax())

# Estimate the saved autograd memory for this model
print(
    "Approx max saved activations:",
    counts.max() * 16.027 / 1024,
    "GiB"
)

In [ ]:
from puma_pipeline.stage2.trainer import generate_stage2_oof

FORCE_STAGE2_OOF = False
stage2_oof = generate_stage2_oof(config, force=FORCE_STAGE2_OOF)
print('Stage-2 OOF predictions:', stage2_oof)


## 8. Fit risk calibration

Fit the risk model and the global candidate-acceptance threshold from OOF predictions.


In [ ]:
from puma_pipeline.stage2.calibration import calibrate_risk

FORCE_CALIBRATION = False
calibration = calibrate_risk(config, force=FORCE_CALIBRATION)
print('Calibration:', calibration)


## 9. Train the final Stage-2 and tissue models

Train the final models on all training ROIs using the same feature contract.


In [ ]:
from puma_pipeline.stage2.trainer import train_stage2_fold
from puma_pipeline.tissue.trainer import train_tissue

final_stage2 = train_stage2_fold(config, None, resume=True)
final_tissue = train_tissue(config, None, resume=True)
print('Final Stage 2:', final_stage2)
print('Final tissue:', final_tissue)


## 10. Package the model files

Check the model states and config fingerprints, then copy the required deployment files into `models/`.


In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable,
    str(CODE_ROOT / 'scripts' / 'package_deployment.py'),
    '--config', str(RESOLVED_CONFIG),
])
print('Deployment models packaged under:', CODE_ROOT / 'models')


## 11. Check the final outputs

Print the main output paths and confirm that the expected files were created.


In [ ]:
from pathlib import Path

for path in sorted((CODE_ROOT / 'models').glob('*')):
    if path.is_file():
        print(path.name, round(path.stat().st_size / 1024**2, 2), 'MB')

print()
print('Stage 2 is complete. Next: build/test/export the Docker image using the scripts in docker/.')


In [ ]:
from google.colab import runtime
runtime.unassign()